# Annotation Validation — MEVA vs Generated Events (LLM as a judge)

Validates generated CCTV event annotations against matched MEVA reference activities, using a Groq-hosted LLM as a reference-based semantic judge.

**Flow**
1. Run configuration
2. Credentials (environment variables / `.env`)
3. Connect to PostgreSQL
4. Load `bronze.matched_pairs` and parse MEVA references
5. Prepare `bronze.annotations_report`
6. Find rows pending for the current `PROMPT_VERSION`, split into **no reference** and **to judge**
7. Judge prompt
8. Failure handling
9. Build report rows and write
10. Run in batches, writing after every batch
11. Summary

Temporal IoU and spatial IoU are separate stages and are not calculated here.

## 1. Run configuration

- `DRY_RUN = True` builds the report in memory and writes **nothing** to the database (no table changes either).
- `MAX_ROWS` limits how many rows are sent to the LLM. Use a small number to test, `None` for all.
- Change `PROMPT_VERSION` whenever you change the prompt or rubric. A row is only skipped if it already exists **for this version**.

After changing anything here, re-run from this cell (or Run All).

In [1]:
PROMPT_VERSION = "v2.3_semantic_judge"

DRY_RUN = False            # True = no database writes or table changes
MAX_ROWS = None           # rows sent to the LLM this run (None = all pending)
BATCH_SIZE = 20

GROQ_MODEL_DEFAULT = "openai/gpt-oss-120b"   # overridden by env GROQ_MODEL if set
REASONING_EFFORT = "low"      # gpt-oss reasoning tokens count against your limits
MAX_COMPLETION_TOKENS = 4096  # hard cap per call (reasoning + answer)
MATCH_THRESHOLD = 0.75        # llm_match = llm_score >= this

MAX_RETRIES = 4               # per call, for rate limits / timeouts / 5xx
MAX_WAIT_SECONDS = 120        # if Groq asks to wait longer (e.g. daily limit), stop the run cleanly
SLEEP_BETWEEN_CALLS = 2.0     # seconds between batches

REPORT_SCHEMA = "bronze"
REPORT_TABLE = "annotations_report"
V1_PROMPT_VERSION = "v1_explicit_judge"   # label for old rows that have no prompt_version

## 2. Credentials

Required: `DB_USER`, `DB_PASSWORD`, `DB_HOST`, `DB_NAME`, `GROQ_API_KEY`
Optional: `DB_PORT` (default `5432`), `GROQ_MODEL` (default `openai/gpt-oss-120b`)

Lookup order: variables already set in the environment → file at `ENV_FILE` → nearest `.env` in the working directory or a parent → a `.env` near the working directory (up to 5 levels deep). Only `.env` files that contain every required variable are used. Keep `.env` out of version control.

In [2]:
%pip install -q pandas sqlalchemy psycopg2-binary "groq>=0.30" python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import json
import math
import os
import re
import time
from pathlib import Path

import pandas as pd
import groq
from groq import Groq
from dotenv import load_dotenv, dotenv_values, find_dotenv
from sqlalchemy import create_engine, text, URL
from sqlalchemy.dialects.postgresql import insert as pg_insert

REQUIRED_ENV = ["DB_USER", "DB_PASSWORD", "DB_HOST", "DB_NAME", "GROQ_API_KEY"]
ENV_SEARCH_DEPTH = 5
SKIP_DIRS = {"node_modules", "__pycache__", "site-packages", "venv", "env"}


def _has_required(path):
    try:
        values = dotenv_values(path)
    except Exception:
        return False
    return all(values.get(name) for name in REQUIRED_ENV)


def _walk_env_files(root, max_depth):
    root = Path(root)
    base = len(root.parts)
    for dirpath, dirnames, filenames in os.walk(root, onerror=lambda e: None):
        depth = len(Path(dirpath).parts) - base
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS and not d.startswith(".")]
        if depth >= max_depth:
            dirnames[:] = []
        if ".env" in filenames:
            yield Path(dirpath) / ".env"


def find_env_file():
    explicit = os.getenv("ENV_FILE")
    if explicit:
        path = Path(explicit).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"ENV_FILE points to a file that does not exist: {path}")
        return path, "ENV_FILE"

    nearest = find_dotenv(usecwd=True)
    if nearest and _has_required(nearest):
        return Path(nearest), "working directory or a parent"

    candidates = sorted({
        p.resolve() for p in _walk_env_files(Path.cwd().parent, ENV_SEARCH_DEPTH) if _has_required(p)
    })
    if len(candidates) > 1:
        raise RuntimeError(
            "Several .env files contain the required variables:\n  "
            + "\n  ".join(map(str, candidates))
            + "\nSet ENV_FILE to the one to use."
        )
    if candidates:
        return candidates[0], "search near the working directory"
    return None, None


if all(os.getenv(name) for name in REQUIRED_ENV):
    print("Credentials: already set in the environment.")
else:
    env_path, found_by = find_env_file()
    if env_path:
        load_dotenv(env_path, override=False)
        print(f"Credentials: loaded .env from {env_path} (found via {found_by}).")

missing_env = [name for name in REQUIRED_ENV if not os.getenv(name)]
if missing_env:
    raise EnvironmentError(
        "Missing required environment variables: " + ", ".join(missing_env)
        + f". No .env with all of them was found near {Path.cwd()}. "
        "Set them in the environment, add a .env, or set ENV_FILE to its path."
    )

GROQ_MODEL = os.getenv("GROQ_MODEL") or GROQ_MODEL_DEFAULT
print("All required environment variables found. Groq model:", GROQ_MODEL)

Credentials: loaded .env from /kaggle/input/datasets/finegoodokay/config/.env (found via search near the working directory).
All required environment variables found. Groq model: openai/gpt-oss-120b


## 3. Database connection

In [4]:
DATABASE_URL = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.environ["DB_HOST"],
    port=int(os.getenv("DB_PORT") or 5432),
    database=os.environ["DB_NAME"],
)

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

with engine.connect() as conn:
    db_name, db_user = conn.execute(text("SELECT current_database(), current_user;")).fetchone()

print("Connection successful. Database:", db_name, "| User:", db_user)

Connection successful. Database: postgres | User: postgres


In [5]:
with engine.connect() as conn:
    counts = pd.read_sql(text('''
        SELECT 'bronze.events' AS table_name, COUNT(*) AS rows FROM bronze.events
        UNION ALL
        SELECT 'bronze.matched_pairs', COUNT(*) FROM bronze.matched_pairs
        '''), conn)
counts

,table_name,rows
0,bronze.events,1158
1,bronze.matched_pairs,123


In [6]:
with engine.begin() as conn:
    deleted = conn.execute(
        text("DELETE FROM bronze.annotations_report WHERE prompt_version = :pv;"),
        {"pv": "v2_semantic_judge"},
    ).rowcount

print(f"Deleted {deleted} rows with prompt_version = 'v2_semantic_judge'.")

Deleted 0 rows with prompt_version = 'v2_semantic_judge'.


In [7]:
with engine.connect() as conn:
    display(pd.read_sql(text('''
        SELECT prompt_version, COUNT(*) AS rows
        FROM bronze.annotations_report
        GROUP BY prompt_version
    '''), conn))

,prompt_version,rows
0,v2.3_semantic_judge,123


In [8]:
with engine.connect() as conn:
    display(pd.read_sql(text('''
        SELECT table_name, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'bronze' AND table_name IN ('events', 'matched_pairs')
        ORDER BY table_name, ordinal_position
    '''), conn))

,table_name,column_name,data_type
0,events,event_id,text
1,events,video_id,text
2,events,event_name,text
3,events,description,text
4,events,start_frame,integer
5,events,end_frame,integer
6,events,start_seconds,double precision
7,events,end_seconds,double precision
8,events,source_status,text
9,events,actions,jsonb


## 4. Load matched pairs and parse MEVA references

Each `meva_reference` is one or more fixed MEVA activity sentences joined by `;` (e.g. `a vehicle makes a U-turn; a vehicle stops`).

A reference counts as **empty** (no MEVA reference) if it is `NULL`, blank, `none`, `[]`, etc., or the pipeline placeholder `(no matching meva ground truth found)`. Those rows are never sent to the LLM.

In [9]:
matched_pairs = pd.read_sql("SELECT * FROM bronze.matched_pairs ORDER BY id;", engine)
print("Matched pairs:", len(matched_pairs))

EMPTY_REFERENCE_TOKENS = {"", "none", "null", "nan", "n/a", "na", "[]", "{}", "-"}
NO_REFERENCE_PATTERN = re.compile(r"no matching meva", re.IGNORECASE)


def normalize_activity(sentence):
    sentence = re.sub(r"\s+", " ", str(sentence)).strip().strip(".").strip()
    return sentence.lower()


def _is_empty_reference(sentence):
    normalized = normalize_activity(sentence)
    return normalized in EMPTY_REFERENCE_TOKENS or bool(NO_REFERENCE_PATTERN.search(normalized))


def parse_reference(value):
    # Return the list of unique, normalised MEVA activity sentences in a reference.
    if value is None:
        return []
    if isinstance(value, float) and math.isnan(value):
        return []
    parts = list(value) if isinstance(value, (list, tuple, set)) else str(value).split(";")

    activities = []
    for part in parts:
        if _is_empty_reference(part):
            continue
        activity = normalize_activity(part)
        if activity not in activities:
            activities.append(activity)
    return activities


reference_counts = matched_pairs["meva_reference"].apply(parse_reference).str.len()
print("With a MEVA reference:", int((reference_counts > 0).sum()))
print("Without a MEVA reference:", int((reference_counts == 0).sum()))

Matched pairs: 123
With a MEVA reference: 79
Without a MEVA reference: 44


## 5. Prepare `bronze.annotations_report`

Only runs when `DRY_RUN = False`. Safe to re-run.
1. Creates the table if it doesn't exist (same columns as `matched_pairs`, no rows).
2. Adds the report columns if they are missing.
3. Tags old rows without a `prompt_version` as `v1_explicit_judge`.
4. Adds a unique index on `(id, prompt_version)` so an event can't be written twice for one version.

In [10]:
REPORT_COLUMNS = {
    "llm_score": "DOUBLE PRECISION",
    "llm_match": "BOOLEAN",
    "llm_reason": "TEXT",
    "eval_status": "TEXT",
    "prompt_version": "TEXT",
    "model": "TEXT",
    "evaluated_at": "TIMESTAMPTZ",
}
FULL_TABLE = f"{REPORT_SCHEMA}.{REPORT_TABLE}"
UNIQUE_INDEX = f"{REPORT_TABLE}_id_prompt_version_uq"


def report_table_exists():
    query = text(
        "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
        "WHERE table_schema = :s AND table_name = :t);"
    )
    with engine.connect() as conn:
        return bool(conn.execute(query, {"s": REPORT_SCHEMA, "t": REPORT_TABLE}).scalar())


def report_table_columns():
    query = text(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema = :s AND table_name = :t;"
    )
    with engine.connect() as conn:
        return {row[0] for row in conn.execute(query, {"s": REPORT_SCHEMA, "t": REPORT_TABLE})}


def ensure_report_table():
    if not report_table_exists():
        matched_pairs.head(0).to_sql(REPORT_TABLE, engine, schema=REPORT_SCHEMA, index=False)
        print(f"Created {FULL_TABLE}.")

    with engine.begin() as conn:
        for column, sql_type in REPORT_COLUMNS.items():
            conn.execute(text(f'ALTER TABLE {FULL_TABLE} ADD COLUMN IF NOT EXISTS "{column}" {sql_type};'))
        tagged = conn.execute(
            text(
                f"UPDATE {FULL_TABLE} SET prompt_version = :v1, "
                "eval_status = COALESCE(eval_status, 'scored') WHERE prompt_version IS NULL;"
            ),
            {"v1": V1_PROMPT_VERSION},
        ).rowcount
    if tagged:
        print(f"Tagged {tagged} existing rows as prompt_version = '{V1_PROMPT_VERSION}'.")

    try:
        with engine.begin() as conn:
            conn.execute(text(f"CREATE UNIQUE INDEX IF NOT EXISTS {UNIQUE_INDEX} ON {FULL_TABLE} (id, prompt_version);"))
        return True
    except Exception as exc:
        print("WARNING: could not create unique index on (id, prompt_version). "
              "There may be duplicate rows already. Writes will use plain append.\n", exc)
        return False


if DRY_RUN:
    USE_ON_CONFLICT = False
    print("DRY_RUN: table not modified.")
else:
    USE_ON_CONFLICT = ensure_report_table()
    print(f"{FULL_TABLE} ready.")

bronze.annotations_report ready.


## 6. Pending rows, split into no reference / to judge

A row is **pending** if there is no row for its `id` under the current `PROMPT_VERSION`, so new videos in `matched_pairs` are picked up automatically.

- **no reference**: recorded as `eval_status = 'no_reference'` with no score, never sent to the LLM
- **to judge**: sent to the LLM (limited by `MAX_ROWS`)

In [11]:
if not report_table_exists() or "prompt_version" not in report_table_columns():
    pending_pairs = matched_pairs.copy()
else:
    pending_query = text(f'''
        SELECT mp.*
        FROM bronze.matched_pairs mp
        WHERE NOT EXISTS (
            SELECT 1 FROM {FULL_TABLE} ar
            WHERE ar.id = mp.id AND ar.prompt_version = :pv
        )
        ORDER BY mp.id;
    ''')
    with engine.connect() as conn:
        pending_pairs = pd.read_sql(pending_query, conn, params={"pv": PROMPT_VERSION})

pending_pairs["reference_list"] = pending_pairs["meva_reference"].apply(parse_reference)
has_reference = pending_pairs["reference_list"].str.len() > 0

no_reference_pairs = pending_pairs[~has_reference].copy()
to_judge = pending_pairs[has_reference].copy()
if MAX_ROWS is not None:
    to_judge = to_judge.head(MAX_ROWS)

print(f"Pending for '{PROMPT_VERSION}': {len(pending_pairs)}")
print(f"  no MEVA reference (not sent to LLM): {len(no_reference_pairs)}")
print(f"  with reference: {int(has_reference.sum())}  ->  sending this run: {len(to_judge)}")

Pending for 'v2.3_semantic_judge': 0
  no MEVA reference (not sent to LLM): 0
  with reference: 0  ->  sending this run: 0


## 7. Groq LLM judge

The rubric accounts for the fact that the model describes **where things move** (trajectories) while MEVA describes **what happens** (a fixed set of activities). A description that is accurate but only describes movement scores 0.50 instead of being treated as wrong; only contradictions score 0.

The model returns a score and a short reason per id. Output uses Groq **strict JSON schema** mode, so the response is always valid JSON with an allowed score. `llm_match` is computed in Python (`llm_score >= MATCH_THRESHOLD`).

In [12]:
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

ALLOWED_SCORES = ["1.00", "0.75", "0.50", "0.25", "0.00"]

SYSTEM_PROMPT = '''You are evaluating generated CCTV event annotations against MEVA reference annotations.

The generated annotation (name + description) usually describes appearance, position and movement.
The MEVA reference lists one or more fixed activities joined by ";". Compare the annotation with EACH
reference activity and give the score of the BEST-matching one.

Apply these steps in order and stop at the first that applies:

STEP 1 -> 0.00 (contradiction)
  The description explicitly states the OPPOSITE of the same kind of action in the reference:
  exits vs enters (a vehicle or the scene), opens vs closes, picks up vs puts down, loads vs unloads,
  turns left vs turns right.
  NOT a contradiction: an actor being idle, standing, parked or simply present. Other people or vehicles
  in the scene may be doing the reference activity.

STEP 2 -> 0.25 (different actor)
  No reference activity has the same kind of actor as the description.
  Actor = the subject of the activity: "a vehicle ..." is a vehicle activity, "a person ..." is a person
  activity (including person-vehicle interactions like "a person opens a vehicle door").
  If the description covers both people and vehicles, both actor types count.

STEP 3 -> 1.00 (same action)
  The description states a reference action, in the same or equivalent words.

STEP 4 -> 0.75 (clearly implied)
  The description gives SPECIFIC evidence of a reference action in different words, for example:
    "curves around and heads back the way it came" -> makes a U-turn
    "pulls into a space and comes to rest" -> stops
    "walks through the doorway into the building" -> exits the scene through a structure
  The evidence must be in the text. Plausible or possible is NOT enough.

STEP 5 -> 0.50 (same actor, action not described)
  Same actor type, no contradiction, but the reference action is not described or clearly implied.
  Descriptions of movement, position, appearance, standing or being present belong here.

Activity rules (a description only counts for these if it describes the change itself):
- "starts moving": a stationary vehicle begins to move. Moving across/through the frame is NOT starting.
- "stops": a moving vehicle comes to a halt. Parked or stationary the whole time is NOT stopping.
- turns / U-turn: a change of direction. Moving from one side of the frame to another is NOT a turn.
- "reverses": the vehicle backs up.
- "enters the scene through a structure": a person comes OUT of a door/building INTO view.
- "exits the scene through a structure": a person goes INTO a door/building and leaves view.
  Approaching or standing near an entrance is neither.
- carries / picks up / puts down / loads / unloads: the object must be mentioned.
- "appears", "is visible", "is detected", "enters the frame" are NOT activities.

Evaluate each annotation independently. Return one result per id.
reason: at most 15 words; name the reference activity compared and the step applied,
e.g. "vehicle stops: same actor, moving only, no stop described (step 5)".'''

RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "score": {"type": "string", "enum": ALLOWED_SCORES},
                    "reason": {"type": "string"},
                },
                "required": ["id", "score", "reason"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["results"],
    "additionalProperties": False,
}

TOKEN_USAGE = {"calls": 0, "prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

## 8. Failure handling

- `call_with_retry`: retries rate limits, timeouts, connection errors and 5xx with exponential backoff (or Groq's `retry-after`). If Groq asks for a wait longer than `MAX_WAIT_SECONDS` (e.g. daily limit used up), the run stops cleanly; everything already written stays written.
- `validate_response`: ids must be the ones sent, no duplicates, scores from the allowed list. A truncated response (`finish_reason = length`) counts as a failure.
- `judge_rows`: if a batch fails or has missing ids, only those rows are retried, split in halves, down to single rows. Rows that still fail are listed in `failed_df` and stay **pending**, so the next run retries them.

In [13]:
RETRYABLE_ERRORS = tuple(
    err for err in (
        getattr(groq, "RateLimitError", None),
        getattr(groq, "APITimeoutError", None),
        getattr(groq, "APIConnectionError", None),
        getattr(groq, "InternalServerError", None),
    ) if err is not None
)


class StopRun(Exception):
    pass


def retry_after_seconds(exc):
    response = getattr(exc, "response", None)
    headers = getattr(response, "headers", None) or {}
    value = headers.get("retry-after")
    try:
        return float(value) if value is not None else None
    except ValueError:
        return None


def call_groq(batch_df):
    annotations = [
        {
            "id": int(row["id"]),
            "generated_name": row["model_event_name"],
            "generated_description": row["model_description"],
            "meva_reference": "; ".join(row["reference_list"]),
        }
        for _, row in batch_df.iterrows()
    ]
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps({"annotations": annotations}, separators=(",", ":"))},
        ],
        temperature=0,
        reasoning_effort=REASONING_EFFORT,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": "annotation_judgements", "strict": True, "schema": RESPONSE_SCHEMA},
        },
    )

    usage = getattr(response, "usage", None)
    TOKEN_USAGE["calls"] += 1
    for key in ("prompt_tokens", "completion_tokens", "total_tokens"):
        TOKEN_USAGE[key] += getattr(usage, key, 0) or 0

    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise ValueError("Response truncated (finish_reason=length). Lower BATCH_SIZE or raise MAX_COMPLETION_TOKENS.")
    return json.loads(choice.message.content)


def call_with_retry(batch_df):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return call_groq(batch_df)
        except RETRYABLE_ERRORS as exc:
            wait = retry_after_seconds(exc) or min(60, 2 ** attempt)
            if wait > MAX_WAIT_SECONDS:
                raise StopRun(f"Groq asked to wait {wait:.0f}s (likely a daily limit). Stopping; rerun later.") from exc
            if attempt == MAX_RETRIES:
                raise
            print(f"    {type(exc).__name__}; retry {attempt}/{MAX_RETRIES - 1} in {wait:.0f}s")
            time.sleep(wait)


def validate_response(raw, batch_df):
    expected_ids = {int(i) for i in batch_df["id"]}
    valid, problems = {}, []

    results = raw.get("results") if isinstance(raw, dict) else None
    if not isinstance(results, list):
        return {}, expected_ids, ["response has no 'results' list"]

    for item in results:
        if not isinstance(item, dict):
            problems.append("non-object result")
            continue
        try:
            event_id = int(item.get("id"))
        except (TypeError, ValueError):
            problems.append(f"bad id {item.get('id')!r}")
            continue
        if event_id not in expected_ids:
            problems.append(f"unexpected id {event_id}")
            continue
        if event_id in valid:
            problems.append(f"duplicate id {event_id}")
            continue
        if item.get("score") not in ALLOWED_SCORES:
            problems.append(f"id {event_id}: invalid score {item.get('score')!r}")
            continue
        valid[event_id] = item

    return valid, expected_ids - set(valid), problems


def judge_rows(batch_df):
    # Returns (valid judgements by id, failed id -> error message).
    try:
        raw = call_with_retry(batch_df)
        valid, missing, problems = validate_response(raw, batch_df)
    except StopRun:
        raise
    except Exception as exc:
        valid, missing, problems = {}, {int(i) for i in batch_df["id"]}, [f"{type(exc).__name__}: {exc}"]

    if not missing:
        return valid, {}

    if len(batch_df) == 1:
        return valid, {next(iter(missing)): "; ".join(problems) or "no valid judgement returned"}

    retry_df = batch_df[batch_df["id"].astype(int).isin(missing)]
    print(f"    {len(missing)} row(s) missing/invalid; retrying in smaller chunks")
    failed = {}
    middle = max(1, len(retry_df) // 2)
    for part in (retry_df.iloc[:middle], retry_df.iloc[middle:]):
        if part.empty:
            continue
        part_valid, part_failed = judge_rows(part)
        valid.update(part_valid)
        failed.update(part_failed)
    return valid, failed

## 9. Build report rows and write

Report row = the original `matched_pairs` columns + `llm_score`, `llm_match`, `llm_reason`, `eval_status`, `prompt_version`, `model`, `evaluated_at`. Inserts only; an existing `(id, prompt_version)` row is never overwritten.

In [14]:
def build_scored_rows(batch_df, judgements):
    scored_df = batch_df[batch_df["id"].astype(int).isin(judgements)].drop(columns=["reference_list"]).reset_index(drop=True)
    scores = [float(judgements[int(i)]["score"]) for i in scored_df["id"]]
    scored_df["llm_score"] = scores
    scored_df["llm_match"] = [s >= MATCH_THRESHOLD for s in scores]
    scored_df["llm_reason"] = [judgements[int(i)]["reason"] for i in scored_df["id"]]
    scored_df["eval_status"] = "scored"
    scored_df["prompt_version"] = PROMPT_VERSION
    scored_df["model"] = GROQ_MODEL
    scored_df["evaluated_at"] = pd.Timestamp.now(tz="UTC")
    return scored_df


def build_no_reference_rows(df):
    report = df.drop(columns=["reference_list"]).reset_index(drop=True)
    report["llm_score"] = pd.Series([None] * len(report), dtype="float64")
    report["llm_match"] = pd.Series([None] * len(report), dtype="object")
    report["llm_reason"] = "No MEVA reference to compare against."
    report["eval_status"] = "no_reference"
    report["prompt_version"] = PROMPT_VERSION
    report["model"] = None
    report["evaluated_at"] = pd.Timestamp.now(tz="UTC")
    return report


def insert_on_conflict_do_nothing(table, conn, keys, data_iter):
    rows = [dict(zip(keys, values)) for values in data_iter]
    if not rows:
        return 0
    statement = pg_insert(table.table).values(rows).on_conflict_do_nothing(index_elements=["id", "prompt_version"])
    return conn.execute(statement).rowcount


def write_report(report_df):
    if DRY_RUN or report_df.empty:
        return 0
    table_columns = report_table_columns()
    write_columns = [c for c in report_df.columns if c in table_columns]
    dropped = [c for c in report_df.columns if c not in table_columns]
    if dropped:
        print("    WARNING: columns not in report table, not written:", dropped)
    with engine.begin() as conn:
        written = report_df[write_columns].to_sql(
            REPORT_TABLE, conn, schema=REPORT_SCHEMA, if_exists="append", index=False,
            method=insert_on_conflict_do_nothing if USE_ON_CONFLICT else "multi",
        )
    return written or 0

## 10. Run

No-reference rows are recorded first (no LLM call). Then each batch is judged and **written immediately**, so a crash or rate limit loses at most the batch in progress.

In [15]:
run_reports = []
failed_rows = {}
rows_written = 0
stopped_early = None

no_reference_report = build_no_reference_rows(no_reference_pairs)
if not no_reference_report.empty:
    rows_written += write_report(no_reference_report)
    run_reports.append(no_reference_report)
    print(f"No-reference rows recorded: {len(no_reference_report)}")

for start in range(0, len(to_judge), BATCH_SIZE):
    batch = to_judge.iloc[start:start + BATCH_SIZE]
    print(f"Batch rows {start + 1}-{start + len(batch)} of {len(to_judge)}")
    try:
        judgements, failed = judge_rows(batch)
    except StopRun as exc:
        stopped_early = str(exc)
        print("  STOPPED:", exc)
        break

    batch_report = build_scored_rows(batch, judgements)
    if not batch_report.empty:
        rows_written += write_report(batch_report)
        run_reports.append(batch_report)
    failed_rows.update(failed)
    print(f"  judged {len(judgements)}, failed {len(failed)}")

    if start + BATCH_SIZE < len(to_judge):
        time.sleep(SLEEP_BETWEEN_CALLS)

annotations_report = pd.concat(run_reports, ignore_index=True) if run_reports else pd.DataFrame()
failed_df = pd.DataFrame([{"id": i, "error": e} for i, e in failed_rows.items()], columns=["id", "error"])

print("\n--- Run complete ---")
print("Mode:", "DRY RUN (nothing written)" if DRY_RUN else "WRITE")
print("Report rows built this run:", len(annotations_report))
print("Rows written to database:", rows_written)
print("Failed (left pending, retried next run):", len(failed_df))
if stopped_early:
    print("Stopped early:", stopped_early)
print("Token usage:", TOKEN_USAGE)
failed_df


--- Run complete ---
Mode: WRITE
Report rows built this run: 0
Rows written to database: 0
Failed (left pending, retried next run): 0
Token usage: {'calls': 0, 'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}


,id,error


## 11. Summary

Uses every row for the current `PROMPT_VERSION` in the database, or only this run's rows when `DRY_RUN = True`. Averages are over rows **with** a MEVA reference; no-reference rows are counted separately.

A lot of 0.50 scores means descriptions are consistent with MEVA but describe movement rather than the MEVA action.

In [16]:
if DRY_RUN or not report_table_exists():
    summary_source = annotations_report.copy()
else:
    with engine.connect() as conn:
        summary_source = pd.read_sql(
            text(f"SELECT * FROM {FULL_TABLE} WHERE prompt_version = :pv;"), conn, params={"pv": PROMPT_VERSION}
        )

if summary_source.empty:
    print("Nothing to summarise yet.")
    summary_overview = score_distribution = pd.DataFrame()
else:
    scored = summary_source[summary_source["eval_status"] == "scored"]
    no_ref = summary_source[summary_source["eval_status"] == "no_reference"]
    summary_overview = pd.DataFrame([
        ("Events evaluated", len(summary_source)),
        ("With MEVA reference (judged)", len(scored)),
        ("No MEVA reference (reported separately)", len(no_ref)),
        ("Mean llm_score (judged only)", round(float(scored["llm_score"].mean()), 3) if len(scored) else None),
        (f"Match rate (score >= {MATCH_THRESHOLD})", round(float(scored["llm_match"].astype(bool).mean()), 3) if len(scored) else None),
    ], columns=["metric", "value"])

    score_distribution = (
        scored["llm_score"].value_counts().reindex([1.0, 0.75, 0.5, 0.25, 0.0], fill_value=0)
        .rename_axis("llm_score").reset_index(name="rows")
    )
    score_distribution["share"] = (score_distribution["rows"] / max(len(scored), 1)).round(3)

print(f"Summary for prompt_version = '{PROMPT_VERSION}'" + (" (this run only, dry run)" if DRY_RUN else ""))
summary_overview

Summary for prompt_version = 'v2.3_semantic_judge'


,metric,value
0,Events evaluated,123.000
1,With MEVA reference (judged),79.000
2,No MEVA reference (reported separately),44.000
3,Mean llm_score (judged only),0.443
4,Match rate (score >= 0.75),0.000


In [17]:
score_distribution

,llm_score,rows,share
0,1.00,0,0.000
1,0.75,0,0.000
2,0.50,62,0.785
3,0.25,16,0.203
4,0.00,1,0.013


In [18]:
# Row-level view of this run (judged rows only; no-reference rows are counted)
preview_columns = [
    "id", "model_event_name", "model_description", "meva_reference",
    "llm_score", "llm_match", "llm_reason", "eval_status",
]
if annotations_report.empty:
    print("No rows built this run.")
else:
    scored_preview = annotations_report[annotations_report["eval_status"] == "scored"]
    n_no_ref = int((annotations_report["eval_status"] == "no_reference").sum())
    print(f"Judged rows this run: {len(scored_preview)} | No-reference rows (not shown): {n_no_ref}")
    display(scored_preview[[c for c in preview_columns if c in scored_preview.columns]])

No rows built this run.


In [19]:
# Row counts per prompt version in the database
if report_table_exists() and "prompt_version" in report_table_columns():
    with engine.connect() as conn:
        display(pd.read_sql(text(f'''
            SELECT prompt_version, eval_status, COUNT(*) AS rows, ROUND(AVG(llm_score)::numeric, 3) AS avg_llm_score
            FROM {FULL_TABLE}
            GROUP BY prompt_version, eval_status
            ORDER BY prompt_version, eval_status;
        '''), conn))
else:
    print("Report table not created yet (run once with DRY_RUN = False).")

,prompt_version,eval_status,rows,avg_llm_score
0,v2.3_semantic_judge,no_reference,44,NaN
1,v2.3_semantic_judge,scored,79,0.443


- events_not_in_matched_pairs is the number of events that still need the matching stage before this notebook can evaluate them. missing_by_video shows which videos they come from, which usually tells you straight away that "the new videos haven't been matched yet".
- Duplicates: If events_rows is greater than events_unique, the same event is stored more than once in events. If matched_pairs_rows is greater than matched_pairs_unique_events, some events have more than one pair, and each pair gets judged separately. Either one is worth knowing before you report numbers.

In [20]:
with engine.connect() as conn:
    coverage = pd.read_sql(text('''
        SELECT
            (SELECT COUNT(*) FROM bronze.events)                                  AS events_rows,
            (SELECT COUNT(DISTINCT event_id) FROM bronze.events)                  AS events_unique,
            (SELECT COUNT(*) FROM bronze.matched_pairs)                           AS matched_pairs_rows,
            (SELECT COUNT(DISTINCT event_id) FROM bronze.matched_pairs)           AS matched_pairs_unique_events,
            (SELECT COUNT(*) FROM bronze.events e
             WHERE NOT EXISTS (SELECT 1 FROM bronze.matched_pairs mp
                               WHERE mp.event_id = e.event_id))                   AS events_not_in_matched_pairs
    '''), conn)

    missing_events = pd.read_sql(text('''
        SELECT e.event_id, e.video_id, e.event_name, e.start_seconds, e.end_seconds, e.source_status
        FROM bronze.events e
        WHERE NOT EXISTS (
            SELECT 1 FROM bronze.matched_pairs mp WHERE mp.event_id = e.event_id
        )
        ORDER BY e.video_id, e.start_seconds
    '''), conn)

    missing_by_video = pd.read_sql(text('''
        SELECT e.video_id, COUNT(*) AS events_not_in_matched_pairs
        FROM bronze.events e
        WHERE NOT EXISTS (
            SELECT 1 FROM bronze.matched_pairs mp WHERE mp.event_id = e.event_id
        )
        GROUP BY e.video_id
        ORDER BY events_not_in_matched_pairs DESC
    '''), conn)

display(coverage)
display(missing_by_video)
missing_events

,events_rows,events_unique,matched_pairs_rows,matched_pairs_unique_events,events_not_in_matched_pairs
0,1158,1158,123,123,1060


,video_id,events_not_in_matched_pairs
0,699c11987bd0696d1a1f688e51709902519e6aac5ff216...,64
1,17d6d8158897d785d58db8e84331a043b365a47a7b3230...,53
2,4fe60946f13083cde18f508595d7d12a5253ead5c5ab21...,51
3,71761b4c6414efe884eb9d8bd6f6e0343d47a4157abc15...,43
4,ef4c656a4e2d8be0a1c83aab3d61a678707b8865ce34ab...,41
...,...,...
61,dc92acaefb538ef82b4f8c8ba59df47e71eec4b724f771...,2
62,755282471b4116e63933b7a9715da671161a53ad2c491e...,1
63,f749d8d43789a3ce1dcfd1ece79ad7229fed1e6c653342...,1
64,a4157aeed1a95a2b7289d088010034301c25e5be71269c...,1


,event_id,video_id,event_name,start_seconds,end_seconds,source_status
0,37a86dd17d15c7afc82edfcb50a82b5d0e123a9d78fcd5...,0045ae35c1de0cc90b4d6e94d5a42352019d513bc54d48...,Person walks past a table with a chair,69.867,77.233,generated
1,d99a1cfd53a6be910c044ab80b81240ddda2acd994ced9...,0045ae35c1de0cc90b4d6e94d5a42352019d513bc54d48...,Person walks onto a basketball court,69.867,77.233,generated
2,41f8ad63f89531c115b5de88a237e508a49eae0d4c4445...,0208e45b75fdda071497fbae9a8e29ed9547c70b992467...,Person walks down a hallway,73.967,81.283,generated
3,b4e91a0191c197bc7b5111c6332d75651fdf642d45dd76...,0208e45b75fdda071497fbae9a8e29ed9547c70b992467...,Person walks down a hallway,81.283,88.600,generated
4,3de78aec0146f3d3dc42c01e54f6374324579dcc132dcd...,0208e45b75fdda071497fbae9a8e29ed9547c70b992467...,Person walks down a hallway,113.733,124.700,generated
...,...,...,...,...,...,...
1055,6f54f81e0670874845cece046f1bc2c451846e296fad29...,ff1f57917d79607fe3946348d7de70c7f65abee89796e0...,Two people enter the room from a doorway,207.900,216.011,generated
1056,c145907772e5a44593361dbab6740348fd1bd0de93e65d...,ff1f57917d79607fe3946348d7de70c7f65abee89796e0...,Two people walk through the room,216.011,224.122,generated
1057,fa141ef1c15ef69301ea28f9085042a137c0ae415a7e03...,ff1f57917d79607fe3946348d7de70c7f65abee89796e0...,Person walks across the room,224.122,232.233,generated
1058,4dfc07b0ca454e29ea475d4bc1ae1ade8180feb15936fb...,ff1f57917d79607fe3946348d7de70c7f65abee89796e0...,Person walks across the room,235.567,240.233,generated
